# 02 · Models and metric

The shared definitions used by notebooks 03 and 04: the experiment settings, the transformer, the n-gram reference model and the fixed-target BPC metric.

In [ ]:
%run 01_prepare_data.ipynb

In [ ]:
import math
from collections import Counter
from dataclasses import dataclass

import numpy as np
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
VOCAB_SIZE = len(vocab)
EVAL_START = 256
CHECKPOINT_DIR = Path("../checkpoints")
RUNS_DIR = RESULTS_DIR / "runs"
RUNS_DIR.mkdir(exist_ok=True)

## Experiment settings

Only `context_len` changes between conditions. Every step sees 16,384 training characters, so the batch size is 512, 128 or 64.

In [ ]:
@dataclass
class Config:
    context_len: int
    seed: int
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 192
    dropout: float = 0.1
    chars_per_step: int = 16384
    max_steps: int = 3000
    lr: float = 1e-3
    min_lr: float = 1e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0
    eval_interval: int = 250
    eval_batches: int = 20

    @property
    def batch_size(self):
        return self.chars_per_step // self.context_len

    @property
    def run_name(self):
        return f"ctx{self.context_len}_seed{self.seed}"

## Transformer

A GPT-style decoder: pre-LayerNorm blocks, causal self-attention, a GELU MLP, learned position embeddings, and output weights tied to the token embeddings.

The position table is created last, and the seed is reset just before the weights are initialised. With the same seed, every weight except the position table therefore starts identical at every context length.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.out = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = config.dropout
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        batch, length, width = x.shape
        q, k, v = (t.view(batch, length, self.n_head, -1).transpose(1, 2) for t in self.qkv(x).split(width, dim=2))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.attn_dropout if self.training else 0.0)
        return self.resid_dropout(self.out(y.transpose(1, 2).reshape(batch, length, width)))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp_norm = nn.LayerNorm(config.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.mlp(self.mlp_norm(x))


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, std=0.02)
    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


class CharTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, config.n_embd)
        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, VOCAB_SIZE, bias=False)
        self.head.weight = self.token_embedding.weight
        self.position_embedding = nn.Embedding(config.context_len, config.n_embd)
        torch.manual_seed(config.seed)
        self.apply(init_weights)

    def forward(self, ids):
        positions = torch.arange(ids.size(1), device=ids.device)
        x = self.embedding_dropout(self.token_embedding(ids) + self.position_embedding(positions))
        return self.head(self.final_norm(self.blocks(x)))

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

In [ ]:
for context_len in [32, 128, 256]:
    print(f"context {context_len}: {CharTransformer(Config(context_len, seed=0)).num_params():,} parameters")

short_model = CharTransformer(Config(32, seed=0)).state_dict()
long_model = CharTransformer(Config(256, seed=0)).state_dict()
assert all(torch.equal(short_model[name], long_model[name]) for name in short_model if name != "position_embedding.weight")
print("Same seed gives identical starting weights apart from the position table")

## N-gram reference

A character n-gram with interpolated absolute-discounting smoothing. For a history *h* and next character *c*:

P_n(c | h) = max(count(h, c) − d, 0) / count(h) + d · types(h) / count(h) · P_{n−1}(c | shorter h)

types(h) is the number of distinct characters seen after *h*. Histories never seen in training fall back fully to the shorter history. The unigram level uses add-one smoothing, so no character gets probability zero.

In [ ]:
class NgramModel:
    def __init__(self, text, max_order):
        self.max_order = max_order
        self.unigram_counts = Counter(text)
        self.total_chars = len(text)
        self.ngram_counts, self.history_counts, self.history_types = {}, {}, {}
        for order in range(2, max_order + 1):
            ngrams = Counter(text[i:i + order] for i in range(len(text) - order + 1))
            history_counts, history_types = Counter(), Counter()
            for ngram, count in ngrams.items():
                history_counts[ngram[:-1]] += count
                history_types[ngram[:-1]] += 1
            self.ngram_counts[order] = ngrams
            self.history_counts[order] = history_counts
            self.history_types[order] = history_types

    def probability_by_order(self, history, char, discount):
        probability = (self.unigram_counts[char] + 1) / (self.total_chars + VOCAB_SIZE)
        probabilities = [probability]
        for order in range(2, self.max_order + 1):
            context = history[-(order - 1):]
            history_count = self.history_counts[order][context]
            if history_count:
                discounted = max(self.ngram_counts[order][context + char] - discount, 0) / history_count
                backoff_weight = discount * self.history_types[order][context] / history_count
                probability = discounted + backoff_weight * probability
            probabilities.append(probability)
        return probabilities

## Metric: fixed-target bits-per-character

Every model scores the same targets: every character from position 257 of a split onward. A model with context length L predicts each target from the L characters before it, and only the last position of each window is scored. BPC = cross-entropy in nats / ln 2.

In [ ]:
def ngram_bpc_by_order(model, split_text, discount):
    bits = np.zeros(model.max_order)
    for t in range(EVAL_START, len(split_text)):
        history = split_text[t - model.max_order + 1:t]
        bits -= np.log2(model.probability_by_order(history, split_text[t], discount))
    return bits / (len(split_text) - EVAL_START)


@torch.no_grad()
def transformer_bpc(model, data, context_len, batch_size=512):
    model.eval()
    targets = torch.arange(EVAL_START, len(data))
    window = torch.arange(-context_len, 0)
    total_nats = 0.0
    for batch_targets in targets.split(batch_size):
        logits = model(data[batch_targets[:, None] + window].to(DEVICE))[:, -1]
        total_nats += F.cross_entropy(logits, data[batch_targets].to(DEVICE), reduction="sum").item()
    return total_nats / len(targets) / math.log(2)